In [1]:
#!/usr/bin/env python

import os

from deepeval.models import OllamaModel
from dotenv import load_dotenv

load_dotenv()

os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "1"
os.environ["DEEPEVAL_DISABLE_TIMEOUTS"] = "1"
JUDGE = "gemma4:e2b"

# Configure Ollama as the LLM judge
JUDGE = OllamaModel(
    model=JUDGE,
    base_url="http://localhost:11434",
    temperature=0.0
)

print("Judge model:", JUDGE)
print("Inference engine: Ollama")
print("LLM judge initialized")

Judge model: <deepeval.models.llms.ollama_model.OllamaModel object at 0x7ac9642733b0>
Inference engine: Ollama
LLM judge initialized


In [2]:
from deepeval.test_case import LLMTestCase

test_case = LLMTestCase(
    input = "How long do refunds take?",
    actual_output = "Refunds are processed within 7 business days",
    expected_output = "Refunds take 7 business days",
    retrieval_context = ["Refunds are processed within 7 business days for orders under 30 days old."]
)

print("input:", test_case.input)
print("output:", test_case.actual_output)
print("retrieval_context:", test_case.retrieval_context)

input: How long do refunds take?
output: Refunds are processed within 7 business days
retrieval_context: ['Refunds are processed within 7 business days for orders under 30 days old.']


In [3]:
from deepeval.metrics import AnswerRelevancyMetric

metric = AnswerRelevancyMetric(threshold=0.7,model=JUDGE)

metric.measure(test_case)

print("score :", metric.score)
print("Passed :", metric.is_successful())
print("Reason :", metric.reason)

Output()

score : 1.0
Passed : True
Reason : The score is 1.00 because the response perfectly and directly answers the user's question about refund timelines. Since there are no irrelevant statements, the score cannot be higher!


In [4]:
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig
from deepeval.metrics import FaithfulnessMetric

good_case = LLMTestCase(
    input="How long do refunds take?",
    actual_output="Refunds are processed within 7 business days",
    retrieval_context=[
        "Refunds are processed within 7 business days for orders under 30 days old."
    ]
)

bad_case = LLMTestCase(
    input="How long do refunds take?",
    actual_output="We refund instantly by UPI, and we also give free vouchers",
    retrieval_context=[
        "Refunds are processed within 7 business days for orders under 30 days old."
    ]
)

metrics = [
    AnswerRelevancyMetric(
        threshold=0.7,
        model=JUDGE,
        async_mode=False
    ),
    FaithfulnessMetric(
        threshold=0.7,
        model=JUDGE,
        async_mode=False
    )
]

results = evaluate(
    test_cases=[good_case, bad_case],
    metrics=metrics,
    async_config=AsyncConfig(run_async=False)
)

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gemma4:e2b (Ollama), strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Faithfulness Metric! (using gemma4:e2b (Ollama), strict=False, 
async_mode=False)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:            How long do refunds take?                                                              │
│  │     Actual Output:    We refund instantly by UPI, and we also give free vouchers                             │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric           ┃ Score ┃ Threshold ┃ Reason                                                    │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Answer Relevancy │ 0.50  │ 0.70      │ The score is 0.50 because while the statement addresses   │
│              │                  │       │           │ the topic of refunds, it focuses on the method of         │
│              │                  │       │           │ refund (UPI) rather than directly answering the           │
│              │                  │       │           │ specific question about the duration of the refund        │
│              │                  │       │           │ process.                                                  │
│        PASS  │ Faithfulness     │ 1.00  │ 0.70      │ The score is 1.00 because there are no contradi...        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                     ┃ Average Score         ┃ Pass Rate                                     ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Answer Relevancy           │ 0.75                  │ 50.00% | passed=1 | failed=1                  │ 2         │
│  Faithfulness               │ 1.00                  │ 100.00% | passed=2 | failed=0                 │ 2         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=10205286;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

Test run saved at report/test_run_20261108_164159.json

✓ Evaluation completed 🎉! (time taken: 44.07s | token cost: None)
» Test Results (2 total tests):
   » Pass Rate: 50.0% | Passed: 1 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

RAG App with Deepeval

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import ChatOllama, OllamaEmbeddings

documents = [
    "Refunds are processed within 7 business days for orders placed under 30 days ago.",
    "Standard shipping takes 3 to 5 business days. Express shipping takes 1 business day.",
    "Our support team is available Monday to Friday, 9am to 6pm IST.",
    "You can cancel a subscription anytime from Settings. Billing stops at the end of the cycle.",
]

embeddings = OllamaEmbeddings(model="embeddinggemma-2:latest")
vector_store = InMemoryVectorStore.from_texts(documents, embedding=embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

llm = ChatOllama(model="bonsai-2:9b", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a support assistant. Answer ONLY using this context: {context}"),
    ("human", "{question}"),
])

print("RAG app ready")

RAG app ready


In [6]:
questions = [
    "How long do refunds take?",
    "How fast is express shipping?",
    "When is support available?"
]

expected = [
    "Refunds are processed within 7 business days",
    "Express shipping takes 1 business days",
    "support is available monday to friday, 9am to 6pm IST."
]

test_cases = []

for question, expected_answer in zip(questions,expected):

    docs = retriever.invoke(question)

    contexts= []
    for doc in docs:
        contexts.append(doc.page_content)

    messages = prompt.format_messages(context = " ".join(contexts), question = question)

    answer = llm.invoke(messages).content

    test_cases.append(LLMTestCase(
        input = question,
        actual_output = answer,
        expected_output = expected_answer,
        retrieval_context = contexts,
    ))

    print("done", question)

done How long do refunds take?
done How fast is express shipping?
done When is support available?


In [7]:
from deepeval.metrics import ContextualPrecisionMetric, ContextualRecallMetric

rag_metrics = [
    AnswerRelevancyMetric(threshold=0.7,model=JUDGE),
    FaithfulnessMetric(threshold=0.7,model=JUDGE),
    ContextualRecallMetric(threshold=0.7,model=JUDGE),
    ContextualPrecisionMetric(threshold=0.7,model=JUDGE)
]

results = evaluate(test_cases=test_cases, metrics=rag_metrics,async_config=AsyncConfig(run_async=False))

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gemma4:e2b (Ollama), strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Faithfulness Metric! (using gemma4:e2b (Ollama), strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using gemma4:e2b (Ollama), strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using gemma4:e2b (Ollama), strict=False, 
async_mode=False)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 4 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 4 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_2 (Passed 4 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                         ┃ Average Score        ┃ Pass Rate                                   ┃ Total    │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━ │
│  Answer Relevancy               │ 1.00                 │ 100.00% | passed=3 | failed=0               │ 3        │
│  Faithfulness                   │ 1.00                 │ 100.00% | passed=3 | failed=0               │ 3        │
│  Contextual Recall              │ 1.00                 │ 100.00% | passed=3 | failed=0               │ 3        │
│  Contextual Precision           │ 1.00                 │ 100.00% | passed=3 | failed=0               │ 3        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=10205288;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

Test run saved at report/test_run_20261108_164502.json

✓ Evaluation completed 🎉! (time taken: 146.74s | token cost: None)
» Test Results (3 total tests):
   » Pass Rate: 100.0% | Passed: 3 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

4.1 - G-Eval: describe the rule in plain English
The easiest one. You write the criteria as a sentence, and an LLM scores against it.

evaluation_params tells it which fields to look at.

In [8]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams

politeness = GEval(
    name="Politeness",
    criteria="Evaluate whether the answer is polite and professional in tone.",
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT
    ],
    threshold=0.7,
    model=JUDGE,
    async_mode=AsyncConfig(run_async=False)
)

rude_case = LLMTestCase(
    input="Where is my order?",
    actual_output="How should I know? Check it yourself."
)

import asyncio


async def run_evaluation():
    await politeness.a_measure(rude_case)

    print("Score:", politeness.score)
    print("Passed:", politeness.is_successful())
    print("Reason:", politeness.reason)

await run_evaluation()

Output()

Score: 0.0
Passed: False
Reason: The actual output fails to meet the requirement for a polite and professional tone. The response uses confrontational language ('Check it yourself') instead of providing helpful information or a polite response to the inquiry about the order status.


In [9]:
from deepeval.metrics import GEval
from deepeval.test_case import SingleTurnParams

politeness = GEval(
    name = "politeness",
    evaluation_steps = [
        "Check whether the answer mentions refunds at all.",
        "If it does, check that it also states the 30 day limit.",
        "Penalise heavily if a refund is promised with no mention of the limit.",
    ],
    evaluation_params = [SingleTurnParams.INPUT,SingleTurnParams.ACTUAL_OUTPUT],
    threshold=0.7,
    model=JUDGE
)

rude_case = LLMTestCase(
    input="Can I get a refund?",
    actual_output="Yes, of course. We will refund you straight away.",
)

politeness.measure(rude_case)

print("score :", politeness.score)
print("reason :", politeness.reason)

Output()

score : 0.1
reason : The response promises a refund but fails to mention the required 30-day limit, which is a mandatory condition according to the evaluation steps.


4.2 - DAG: a decision tree you control
G-Eval is one judgement call, so the score can wobble between runs. A DAG breaks the decision into small yes or no steps that you define.

The four building blocks:

Node	What it does
BinaryJudgementNode	asks a yes or no question. Verdicts must be True and False
NonBinaryJudgementNode	asks a multiple choice question. Verdicts are strings
TaskNode	extracts or reshapes something first, before any judging
You build it top down with two methods:

Method	Use on	Meaning
.add_verdict(verdict=..., score=N)	a judgement node	this answer ends here with score N
.add_verdict(verdict=..., then=node)	a judgement node	this answer continues to another question
.add_node(child)	a TaskNode	hand the extracted result to the next node
The tree we are about to build:

                 Does it mention a number of days?
                          /              \
                       True             False
                        /                  \
        Does it say 7 business days?      score 0
              /            \
           True           False
            /                \
        score 10          score 2
Two things that trip people up.

A BinaryJudgementNode needs verdict=True and verdict=False. The strings "yes" and "no" raise ValueError: All children BinaryJudgementNode must have a boolean verdict.
Older tutorials build nodes with children=[VerdictNode(...)]. That still works but is deprecated and prints a warning. Use .add_verdict() as below.

In [10]:
from deepeval.metrics import DAGMetric
from deepeval.metrics.dag import DeepAcyclicGraph, BinaryJudgementNode


says_seven_days = BinaryJudgementNode(criteria = "Does the answer says 7 business days")
says_seven_days.add_verdict(verdict=True, score=10)
says_seven_days.add_verdict(verdict=False, score=2)

mentions_days = BinaryJudgementNode(criteria = "Does the answer mention a number of days at all?")
mentions_days.add_verdict(verdict=True, then = says_seven_days)
mentions_days.add_verdict(verdict=False, score=0)

dag = DeepAcyclicGraph(root_nodes=[mentions_days])

refund_dag = DAGMetric(name="Refund answer check", dag=dag, threshold = 0.7, model = JUDGE)

print(" DAG Metric Ready", refund_dag.name)

 DAG Metric Ready Refund answer check


In [11]:
answers = [
    "Refunds are processed within 7 business days.",     
    "Refunds usually take about 20 days.",               
    "We will sort out your refund soon.",                
]

for answer in answers:
    case = LLMTestCase(input="How long do refund take?", actual_output=answer)

    refund_dag.measure(case)

    print("Answer :", answer)
    print("passed :", refund_dag.is_successful())
    print("score :", refund_dag.score)
    

Output()

Output()

Answer : Refunds are processed within 7 business days.
passed : False
score : 0.0


Output()

Answer : Refunds usually take about 20 days.
passed : False
score : 0.0


Answer : We will sort out your refund soon.
passed : False
score : 0.0


4.3 - Arena G-Eval: which output is better?
The other two ask "is this good?". Arena asks "which of these is better?"

Use it to compare two prompts, two models, or two versions of your app on the same input.

Two rules the contestants must follow: every contestant needs a unique name, and they must all share the same input. You are comparing answers to one question.

Judging "A or B" is far more reliable than scoring each one alone, because two mediocre answers can both land on 0.75 and tell you nothing.

In [12]:
from deepeval.metrics import ArenaGEval
from deepeval.test_case import ArenaTestCase
from deepeval.test_case.arena_test_case import Contestant

question = "How long do refunds take?"

arena_case = ArenaTestCase(contestants=[
    Contestant(
        name="Short prompt",
        test_case=LLMTestCase(input=question, actual_output="7 business days."),
    ),
    Contestant(
        name="Detailed prompt",
        test_case=LLMTestCase(
            input=question,
            actual_output="Refunds are processed within 7 business days, "
                                    "for orders placed under 30 days ago.",
        ),
    ),
])





comparison = ArenaGEval(
    name="Most helpful answer",
    criteria="Pick the answer that is clearer and gives the customer more useful detail, "
                                    "without adding anything untrue.",
                                    
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT],
    model=JUDGE,
)

comparison.measure(arena_case)

print("winner:", comparison.winner)
print("reason:", comparison.reason)


Output()

winner: Detailed prompt
reason: Bob is selected because his actual output provides the highest level of usefulness and clarity by answering the question while also including necessary constraints. While Short prompt provided a clear answer of "7 business days," Detailed prompt's response is more useful as it specifies the condition for that refund period (orders placed under 30 days ago), thereby providing a more complete and actionable answer to the input "How long do refunds take?"


In [13]:
from deepeval import compare

win_counts = compare(test_cases=[arena_case], metric=comparison)

print(win_counts)

Output()

🎉 Arena completed! (time taken: 13.98s | token cost: 0.0 USD)
🏆 Results (1 total test cases):
    » Detailed prompt: 1 wins

================================================================================

» Want to share experiments with your team? ❤️ 🏟️
  » Run 'deepeval login' to analyze and save arena results on Confident AI.

{'Detailed prompt': 1}
